# Phase 3: Feature Engineering
## Avoidable Emergency Department Utilization Navigator

This notebook generates member-level aggregated utilization features from cleaned CMS Medicare Enrollment and FFS Claims datasets (`beneficiary_clean.csv`, `inpatient_clean.csv`, `outpatient_clean.csv`).

### Key Requirements:
1. **Member Profile Features**: Age, Gender, Dual Eligibility Months, Chronic Condition Count.
2. **Utilization Features**: ED Visit Count (`REV_CNTR == '0450'`), Inpatient Visit Count, Outpatient Utilization Count.
3. **Cost Features**: Total Claim Payment Amount, Average Claim Cost, ED-related Spend.
4. **Provider Features**: Distinct Provider Count (`PRVDR_NUM`).
5. **Temporal Utilization Features**: Utilization Span Days, Active Utilization Months.
6. **Output Destination**: `processed_data/utilization_features.csv`.

### Step 1: Load Cleaned Datasets

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

processed_dir = '../processed_data'

df_bene = pd.read_csv(os.path.join(processed_dir, 'beneficiary_clean.csv'), low_memory=False)
df_inp = pd.read_csv(os.path.join(processed_dir, 'inpatient_clean.csv'), low_memory=False)
df_outp = pd.read_csv(os.path.join(processed_dir, 'outpatient_clean.csv'), low_memory=False)

print(f"Beneficiaries: {len(df_bene):,} rows")
print(f"Inpatient Claims: {len(df_inp):,} rows")
print(f"Outpatient Claims: {len(df_outp):,} rows")

### Step 2: Extract Member Profile Features

In [ ]:
df_bene['BENE_ID'] = df_bene['BENE_ID'].astype(str).str.strip()

# Age calculation
if 'AGE_AT_END_REF_YR' in df_bene.columns:
    df_bene['age'] = pd.to_numeric(df_bene['AGE_AT_END_REF_YR'], errors='coerce').fillna(65).astype(int)
else:
    birth_yr = pd.to_datetime(df_bene['BENE_BIRTH_DT'], errors='coerce').dt.year
    df_bene['age'] = (2022 - birth_yr).fillna(65).astype(int)

# Gender mapping
df_bene['gender'] = df_bene['SEX_IDENT_CD'].astype(str).map({'1': 'Male', '2': 'Female'}).fillna('Unknown')

# Dual eligibility months
if 'DUAL_ELGBL_MONS' in df_bene.columns:
    df_bene['dual_eligibility_months'] = pd.to_numeric(df_bene['DUAL_ELGBL_MONS'], errors='coerce').fillna(0).astype(int)
else:
    df_bene['dual_eligibility_months'] = 0

# Chronic condition count
chronic_cols = [c for c in df_bene.columns if c.startswith('SP_') or c.startswith('CHRONIC_')]
df_bene['chronic_condition_count'] = (df_bene[chronic_cols] == 1).sum(axis=1) if chronic_cols else 0

profile_df = df_bene[['BENE_ID', 'age', 'gender', 'dual_eligibility_months', 'chronic_condition_count']].copy()
profile_df.head()

### Step 3: Extract Inpatient & Outpatient Utilization & Cost Features

In [ ]:
# Inpatient Aggregations
df_inp['BENE_ID'] = df_inp['BENE_ID'].astype(str).str.strip()
df_inp['REV_CNTR'] = df_inp['REV_CNTR'].astype(str).str.strip()
df_inp['is_ed'] = df_inp['REV_CNTR'].str.contains('0450|450', na=False)

inp_agg = df_inp.groupby('BENE_ID').agg(
    inpatient_visit_count=('CLM_ID', 'nunique'),
    inp_ed_visits=('CLM_ID', lambda x: df_inp.loc[x.index, 'is_ed'].sum()),
    inpatient_total_cost=('CLM_PMT_AMT', 'sum'),
    inp_ed_cost=('CLM_PMT_AMT', lambda x: df_inp.loc[x.index[df_inp.loc[x.index, 'is_ed']], 'CLM_PMT_AMT'].sum()),
    inp_providers=('PRVDR_NUM', lambda x: set(x.dropna().astype(str)))
).reset_index()

# Outpatient Aggregations
df_outp['BENE_ID'] = df_outp['BENE_ID'].astype(str).str.strip()
df_outp['REV_CNTR'] = df_outp['REV_CNTR'].astype(str).str.strip()
df_outp['is_ed'] = df_outp['REV_CNTR'].str.contains('0450|450', na=False)

outp_agg = df_outp.groupby('BENE_ID').agg(
    outpatient_visit_count=('CLM_ID', 'nunique'),
    outp_ed_visits=('CLM_ID', lambda x: df_outp.loc[x.index, 'is_ed'].sum()),
    outpatient_total_cost=('CLM_PMT_AMT', 'sum'),
    outp_ed_cost=('CLM_PMT_AMT', lambda x: df_outp.loc[x.index[df_outp.loc[x.index, 'is_ed']], 'CLM_PMT_AMT'].sum()),
    outp_providers=('PRVDR_NUM', lambda x: set(x.dropna().astype(str)))
).reset_index()

### Step 4: Merge Member Features & Compute Aggregates

In [ ]:
features = profile_df.merge(inp_agg, on='BENE_ID', how='left').merge(outp_agg, on='BENE_ID', how='left')

fill_zero_cols = ['inpatient_visit_count', 'inp_ed_visits', 'inpatient_total_cost', 'inp_ed_cost',
                  'outpatient_visit_count', 'outp_ed_visits', 'outpatient_total_cost', 'outp_ed_cost']
for col in fill_zero_cols:
    features[col] = features[col].fillna(0)

# Combined Utilization Metrics
features['ed_visit_count'] = (features['inp_ed_visits'] + features['outp_ed_visits']).astype(int)
features['inpatient_visit_count'] = features['inpatient_visit_count'].astype(int)
features['outpatient_visit_count'] = features['outpatient_visit_count'].astype(int)

features['total_claim_payment_amount'] = (features['inpatient_total_cost'] + features['outpatient_total_cost']).round(2)
features['total_ed_related_cost'] = (features['inp_ed_cost'] + features['outp_ed_cost']).round(2)

total_encounters = features['inpatient_visit_count'] + features['outpatient_visit_count']
features['average_claim_cost'] = np.where(total_encounters > 0, (features['total_claim_payment_amount'] / total_encounters).round(2), 0.0)

# Combined Distinct Provider Count
features['inp_providers'] = features['inp_providers'].apply(lambda x: x if isinstance(x, set) else set())
features['outp_providers'] = features['outp_providers'].apply(lambda x: x if isinstance(x, set) else set())
features['provider_count'] = features.apply(lambda r: len(r['inp_providers'].union(r['outp_providers'])), axis=1)

# Drop intermediate set/list columns
features.drop(columns=['inp_providers', 'outp_providers', 'inp_ed_visits', 'inp_ed_cost', 'outp_ed_visits', 'outp_ed_cost'], inplace=True)
features.head(10)

### Step 5: Save & Validate Feature Dataset

In [ ]:
out_path = os.path.join(processed_dir, 'utilization_features.csv')
features.to_csv(out_path, index=False)

print(f"Feature Dataset Saved: {out_path} ({os.path.getsize(out_path)/(1024*1024):.2f} MB)")
print(f"Total Members Aggregated: {len(features):,}")
print(f"Feature Columns ({len(features.columns)}): {list(features.columns)}")